# Batch Model Monitoring Demo - Observe & Query Metrics

Demonstrates how to programmatically query monitoring metrics:
1. **Drift metrics** (`POPULATION_STABILITY_INDEX`) — detect feature distribution shifts
2. **Statistical metrics** (`COUNT`, `COUNT_NULL`, `MIN`, `MAX`, `AVG`) — data quality
3. **Segment-level queries** — per PLAN_TYPE and CONTRACT_TYPE

Available drift metrics: `POPULATION_STABILITY_INDEX`, `JENSEN_SHANNON`, `WASSERSTEIN`, `DIFFERENCE_OF_MEANS`

## Interpreting PSI (Population Stability Index)
| PSI Value | Interpretation |
|-----------|---------------|
| < 0.1 | No significant shift |
| 0.1 – 0.25 | Moderate shift — investigate |
| > 0.25 | Significant shift — action required |

In [ ]:
%%sql -r df_ctx
USE DATABASE ML_DEMOS;
USE SCHEMA BATCH_MONITORING;
USE WAREHOUSE ML_DEMO_WH;

In [ ]:
%%sql -r df_status
-- Check monitor status
DESCRIBE MODEL MONITOR CHURN_MONITOR;

## Drift Metrics

Query PSI (Population Stability Index) for individual features over the monitoring window.

In [ ]:
%%sql -r df_drift_charges
-- PSI drift for MONTHLY_CHARGES (should show increasing drift over days)
SELECT *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CHURN_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'MONTHLY_CHARGES',
    '1 DAY',
    '2025-01-14'::TIMESTAMP_NTZ,
    '2025-01-24'::TIMESTAMP_NTZ,
    NULL
))
ORDER BY 1;

In [ ]:
%%sql -r df_drift_tenure
-- PSI drift for TENURE_MONTHS (shifts starting day 4)
SELECT *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CHURN_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'TENURE_MONTHS',
    '1 DAY',
    '2025-01-14'::TIMESTAMP_NTZ,
    '2025-01-24'::TIMESTAMP_NTZ,
    NULL
))
ORDER BY 1;

In [ ]:
%%sql -r df_drift_score
-- Prediction score distribution drift
SELECT *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CHURN_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'PREDICTION_SCORE',
    '1 DAY',
    '2025-01-14'::TIMESTAMP_NTZ,
    '2025-01-24'::TIMESTAMP_NTZ,
    NULL
))
ORDER BY 1;

## Statistical Metrics

Row volume and data quality over time.

In [ ]:
%%sql -r df_stats
-- Row count per aggregation window
SELECT *
FROM TABLE(MODEL_MONITOR_STAT_METRIC(
    'CHURN_MONITOR',
    'COUNT',
    'MONTHLY_CHARGES',
    '1 DAY',
    '2025-01-14'::TIMESTAMP_NTZ,
    '2025-01-24'::TIMESTAMP_NTZ,
    NULL
))
ORDER BY 1;

## Segment-Level Metrics

Monitor drift independently for different customer segments.

In [ ]:
%%sql -r df_segment_premium
-- Drift for PREMIUM customers only
SELECT *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CHURN_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'MONTHLY_CHARGES',
    '1 DAY',
    '2025-01-14'::TIMESTAMP_NTZ,
    '2025-01-24'::TIMESTAMP_NTZ,
    '{"SEGMENTS": [{"column": "PLAN_TYPE", "value": "PREMIUM"}]}'
))
ORDER BY 1;

In [ ]:
%%sql -r df_segment_mtm
-- Drift for MONTH_TO_MONTH contract customers
SELECT *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CHURN_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'MONTHLY_CHARGES',
    '1 DAY',
    '2025-01-14'::TIMESTAMP_NTZ,
    '2025-01-24'::TIMESTAMP_NTZ,
    '{"SEGMENTS": [{"column": "CONTRACT_TYPE", "value": "MONTH_TO_MONTH"}]}'
))
ORDER BY 1;

## Monitor Management

Suspend, resume, and modify monitor configuration.

In [ ]:
%%sql -r df_suspend
-- Suspend monitoring (e.g., during maintenance)
ALTER MODEL MONITOR CHURN_MONITOR SUSPEND;

-- Check status
DESCRIBE MODEL MONITOR CHURN_MONITOR;

In [ ]:
%%sql -r df_resume
-- Resume monitoring
ALTER MODEL MONITOR CHURN_MONITOR RESUME;

-- Verify resumed
DESCRIBE MODEL MONITOR CHURN_MONITOR;

## Visual Dashboard

For the full visual experience, open Snowsight and navigate to:

**AI & ML → Models → CHURN_PREDICTOR → Monitors → CHURN_MONITOR**

The dashboard provides:
- Time-series charts of PSI per feature
- Prediction score histograms over time
- Row volume trends
- Segment selector to filter by PLAN_TYPE or CONTRACT_TYPE
- Compare button to overlay multiple monitors

## Cleanup (optional)

```sql
DROP MODEL MONITOR IF EXISTS CHURN_MONITOR;
DROP MODEL IF EXISTS CHURN_PREDICTOR;
DROP TABLE IF EXISTS SCORING_DATA;
DROP TABLE IF EXISTS BASELINE_DATA;
DROP TABLE IF EXISTS CUSTOMERS;
DROP TABLE IF EXISTS SUBSCRIPTION_EVENTS;
DROP SCHEMA IF EXISTS ML_DEMOS.BATCH_MONITORING;
```